# AHN window diagnostic — GatedDeltaNet, observe-only (Task #1)

Self-contained. **No repo upload, no GitHub.** The minimal `ahn-mdc` project (branch `saadat-pipeline-validation`, HEAD `9d60bee`) is embedded below as a base64 tarball and reconstructed into `/kaggle/working/ahn-mdc`.

The Juan-approved matched inference config is baked into the bundled `src/ahnexp/models.py::_force_window` (**sliding_window=256, sliding_window_type=fixed, ahn_position=prefix, num_attn_sinks=0**, stale `dy_*` deleted). The diagnostic is **observe-only** — it never writes `model.config`.

## ⚠️ Use a FRESH Kaggle session
The earlier attempt compiled flash-attn from source, exhausted RAM, and left torch/torchvision corrupted. **Start a brand-new notebook (or Factory reset)** — a kernel restart does not undo broken on-disk packages. Then set: Accelerator = **GPU L4** (flash-attn 2.x needs sm_80+; Kaggle T4/P100 fail) · Internet = **On** · Persistence = *Files only*.

## Run order
| step | cell | note |
|---|---|---|
| A — reconstruct project | **1** | seconds |
| B — environment | **2** | ~5–10 min; pins torch 2.6 + **prebuilt** flash-attn wheel, **no compile**; exits non-zero on any import failure |
| **RESTART KERNEL** | — | Run ▸ Restart & clear cell outputs (torch was replaced on disk) |
| C — verify env | **3** | seconds; imports + a real `flash_attn_func` GPU call |
| D — diagnostic | **4** | first run ~15–20 min (6 GB base download + one-time merge); **exactly 2 generations** |
| E — JSON | **5** | seconds |

tarball sha256: `56500983a2950338d00f60c7542630c9082f2e13ddf4b4c0d5a97332dca4afd9`


## A · Cell 1 — reconstruct the minimal project


In [ ]:
import base64, hashlib, io, tarfile, pathlib

ROOT = "/kaggle/working/ahn-mdc"
EXPECT_SHA = "56500983a2950338d00f60c7542630c9082f2e13ddf4b4c0d5a97332dca4afd9"

_BLOB = (
    "..."
    "/v00ANACAA=="
)

blob = base64.b64decode(_BLOB)
assert hashlib.sha256(blob).hexdigest() == EXPECT_SHA, "tarball checksum mismatch"
pathlib.Path(ROOT).mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as tf:
    tf.extractall(ROOT)

got = sorted(str(p.relative_to(ROOT)) for p in pathlib.Path(ROOT).rglob("*") if p.is_file())
print(f"reconstructed {len(got)} files under {ROOT}")

for need in ("config/experiment.yaml", "config/facts.yaml", "src/ahnexp/models.py",
             "src/ahnexp/dataset.py", "src/ahnexp/evaluate.py",
             "scripts/diag_ahn_window.py", "scripts/setup_kaggle.sh"):
    assert (pathlib.Path(ROOT) / need).is_file(), f"missing {need}"

m = (pathlib.Path(ROOT) / "src/ahnexp/models.py").read_text()
assert 'model.config.sliding_window_type = matched["sliding_window_type"]' in m
assert "model.config.num_attn_sinks = 0" in m
assert '("dy_sliding_window", "dy_num_attn_sinks")' in m
s = (pathlib.Path(ROOT) / "scripts/setup_kaggle.sh").read_text()
assert 'TORCH_VER="2.6.0"' in s, 'bundled setup does not pin torch 2.6.0'
assert 'FA_VER="2.8.3.post1"' in s, 'bundled setup does not use the prebuilt flash-attn wheel'
assert 'FLASH_ATTENTION_FORCE_BUILD' not in s, 'bundled setup would compile flash-attn'
print("OK - bundled models.py has the matched-config freeze; setup pins torch 2.6 + prebuilt flash-attn.")


## B · Cell 2 — environment (pin torch 2.6, prebuilt flash-attn, AHN + fla)

Runs the bundled `scripts/setup_kaggle.sh`: pins `torch==2.6.0` from the **cu126** (manylinux_2_28 / CXX11-ABI-TRUE) index, removes torchvision/torchaudio, installs the **prebuilt** `flash_attn-2.8.3.post1` torch2.6 abiTRUE wheel by URL (**no source build**), the Seerkfang `flash-linear-attention` fork, and the AHN package (core only). A pip constraints file keeps torch / transformers / triton fixed. If `fla`, `flash_attn`, or `ahn.transformer.qwen2_ahn` fails to import the script exits non-zero and prints **no** success line.


In [ ]:
import subprocess
rc = subprocess.call(["bash", "/kaggle/working/ahn-mdc/scripts/setup_kaggle.sh"])
print("\nsetup exit code:", rc)
assert rc == 0, "setup failed - see the FAIL lines above. Start a FRESH Kaggle session and retry."
print(">>> RESTART THE KERNEL now (Run -> Restart & clear cell outputs), then run Cell 3. <<<")


## ⚠️ RESTART THE KERNEL NOW

**Run ▸ Restart & clear cell outputs.** `torch` was just replaced on disk (2.10 → 2.6); the running kernel still holds the old one.

After restarting, run **Cell 3, 4, 5**. Do **not** re-run Cell 1 / Cell 2.


## C · Cell 3 — verify the environment (fail fast)


In [ ]:
import importlib, torch, transformers
print("torch        ", torch.__version__, "| cuda", torch.version.cuda,
      "| available", torch.cuda.is_available())
print("gpu          ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("transformers ", transformers.__version__)
assert torch.__version__.startswith("2.6."), (
    f"torch is {torch.__version__} - the kernel still has the old torch; RESTART and re-run Cell 3.")
assert transformers.__version__ == "4.51.0", transformers.__version__
assert torch.cuda.is_available(), "no CUDA GPU - set Accelerator to GPU"
cap = torch.cuda.get_device_capability()
print(f"gpu capability sm_{cap[0]}{cap[1]}")
assert cap[0] >= 8, (
    f"GPU is sm_{cap[0]}{cap[1]} - flash-attn 2.x needs sm_80+ (Ampere/Ada/Hopper). "
    "Kaggle T4/P100 will not work; select the L4 accelerator.")

for mod in ("triton", "fla", "flash_attn", "ahn.transformer.qwen2_ahn"):
    x = importlib.import_module(mod)
    print(f"import {mod:30} OK  {getattr(x, '__version__', '')}")

# prove the flash-attn CUDA extension actually runs (not just imports)
from flash_attn import flash_attn_func
q = torch.randn(1, 8, 2, 16, dtype=torch.float16, device="cuda")
o = flash_attn_func(q, q, q, causal=True)
assert tuple(o.shape) == (1, 8, 2, 16)
print("flash_attn_func on GPU: OK", tuple(o.shape))

# prove the AHN custom Qwen2 classes register
from ahn.transformer.qwen2_ahn import register_customized_qwen2
register_customized_qwen2()
print("register_customized_qwen2: OK")
print("\nENV VERIFIED - safe to run Cell 4.")


## D · Cell 4 — run the observe-only diagnostic

Loads **only GatedDeltaNet**, merges weights once (~6 GB base download, cached), verifies the custom AHN class + `.ahn` params, builds two trajectories straddling W=256, **hard-stops if the recurrent one does not cross W**, then runs **exactly one exact-memory and one recurrent-memory generation**. Observe-only.


In [ ]:
import subprocess, sys
cmd = [sys.executable, "/kaggle/working/ahn-mdc/scripts/diag_ahn_window.py",
       "--repo", "/kaggle/working/ahn-mdc", "--ahn-repo", "/kaggle/working/AHN"]
print(" ".join(cmd), "\n")
rc = subprocess.call(cmd)
print("\ndiagnostic exit code:", rc, "(0 = PASSED, 2 = INCONCLUSIVE, 1 = hard-stop/error)")


## E · Cell 5 — print the diagnostic JSON


In [ ]:
import pathlib
p = pathlib.Path("/kaggle/working/ahn-mdc/outputs/diag_ahn_window.json")
print(p.read_text() if p.is_file() else
      "no JSON - the run hard-stopped early; paste the Cell 4 output above.")


## What to paste back for review

Full output of **Cell 3** (env), **Cell 4** (diagnostic), and the **Cell 5** JSON. Key checks:

* Cell 2: `SETUP OK (torch 2.6.0 cxx11abi=TRUE …)` and `GPU … sm_89`
* Cell 3: `torch 2.6.x`, `flash_attn_func on GPU: OK`, `register_customized_qwen2: OK`
* `[2]` checkpoint pre-override: `sliding_window 256`, `sliding_window_type random`, `ahn_position random`
* `[3]` after `models.load`: `sliding_window 256`, **`sliding_window_type fixed`**, **`ahn_position prefix`**, **`num_attn_sinks 0`**, `dy_* <unset>`, `model.training False`, `effective_window 256`
* `[3b]` `model class ahn.transformer.qwen2_ahn.qwen2_ahn.Qwen2ForCausalLM`; `generation_config.use_cache True`
* `[2] AHN modules` `36 x Qwen2MemDecoderLayer`, `layer[0] .ahn = BaseAHN / fn=GatedDeltaNet`
* `[6]` the two `model_tokens_after_target` (≈49 and ≈2100)
* `HARD GATE` line = `PASS`
* `[9]` both trial rows: `prediction`, `answer_canonical`, `correct`, `malformed`, `abstained`, `n_new_tokens`, **`ahn_layer0_num_cached_tokens`** (0 exact / ~1850+ recurrent), **`ahn_kernel_forward_calls`** (0 exact / ≥1 recurrent)
* `[10]` verdict block + `DIAGNOSTIC PASSED` / `INCONCLUSIVE`

If Cell 4 exits early, paste what printed — the hard gate stopping is a valid result. **Do not run the experimental grid regardless of outcome.**
